In [43]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep
from scipy.stats import ttest_ind

In [44]:
sample_size = 3500
df = pd.read_csv("../data/ab_test_dataset.csv")
df = df.sample(frac=1, random_state=42, ignore_index=True)
control_df = df[df["group"] == "control"].head(sample_size).reset_index(drop=True)
treatment_df = df[df["group"] == "treatment"].head(sample_size).reset_index(drop=True)
sample_size_df = pd.concat([control_df, treatment_df], axis=0, ignore_index=True)

control_AOV = df[df["group"]=="control"]["AOV_col"].head(sample_size)
treatment_AOV = df[df["group"]=="treatment"]["AOV_col"].head(sample_size)

In [45]:
summary = sample_size_df.groupby("group")["converted"].agg(["count", "sum", "mean"]).rename(
    columns={"count": "total_users", "sum": "conversions", "mean": "converion_rate"}
)
print(summary)

           total_users  conversions  converion_rate
group                                              
control           3500          757        0.216286
treatment         3500          863        0.246571


In [46]:
count = [summary.loc["treatment", "conversions"], summary.loc["control", "conversions"]]
nobs = [summary.loc["treatment", "total_users"], summary.loc["control", "total_users"]]

z_stat, p_value = proportions_ztest(count, nobs)
print(f"z-статистика: {z_stat:.3f}")
print(f"p-value: {p_value:.4f}")

z-статистика: 3.004
p-value: 0.0027


In [47]:
ci_low, ci_upp = confint_proportions_2indep(
    count1=summary.loc["treatment", "conversions"], nobs1=summary.loc["treatment", "total_users"],
    count2=summary.loc["control", "conversions"], nobs2=summary.loc["control", "total_users"]
)
print(f"95% доверительный интервал разности конверсий: [{ci_low:.4f}, {ci_upp:.4f}]")

95% доверительный интервал разности конверсий: [0.0105, 0.0500]


In [48]:
t_stat_AOV, p_value_AOV = ttest_ind(control_AOV, treatment_AOV, equal_var=False)
print(f"t-статистика AOV: {t_stat_AOV:.3f}, p-value AOV: {p_value_AOV:.4f}")

t-статистика AOV: -1.529, p-value AOV: 0.1262


# Пояснения к блокноту

Для начала тестируем основную метрику: конверсию. sample_size взят из MDE, но оуркглённый в большую сторону, перемешиваем выборку на всякий случай, используем z-test так как метрика бинарная, из него получаем p-value, в нашем случае < 0.05, что позволяет отклонить нулевую гипотезу и принять альтернативную, дополнительно получаем доверительный интервал разности конверсий, он поможет оценить размер различий между группами.

Теперь оценим guardrail метрику: AOV. Берём тот же sample_size, так как предполагается, что всего было полученно sample_size результатов в каждой группе. Тестируем AOV при помощи t-теста, так как метрика непрерывное число, получаем p-value < 0.05, то есть это метрика статистически значимо не изменилась. Если бы эта guardrail статистически значимо ухудшилась (в данном случае уменшилась), то однозначно рекомендовать бы продукт группы B было бы нельзя, даже при статистически значимом улучшении целевой метрики.